# 04 · Tuning de los tres mejores candidatos

30 configuraciones por cada una de las tres mejores familias **para t+3** (90 nested runs). Búsqueda aleatoria reproducible con ParameterSampler y evaluación explícita en los mismos folds; facilita registrar cada modelo y su evidencia.

In [ ]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image, Markdown
from src.datos import DATA, DOCS, EVIDENCE
from src.datos import execute
HAS_DATA = (DATA / "obt.parquet").exists()
display(Markdown("**OBT real local disponible**" if HAS_DATA else
    "**PENDIENTE:** faltan credenciales Snowflake válidas y snapshot. Este notebook no contiene resultados empíricos del ECU 911."))


## Espacios predefinidos

Regularización y learning rate en escala logarítmica; profundidad, hojas y tamaños en rangos acotados. Se evalúa la mejora sobre el default. Si no supera la mayor SD entre pliegues de ambas versiones, se conserva el default para selección. No se abre test.

|Familia|Espacio disponible|
|---|---|
|Logística|C log 1e-4–1e3; ponderación None/balanced; 500/1000 iteraciones|
|Random forest|80–400 árboles; profundidad None/6/12/20; hoja mínima 1–30; max_features sqrt/.3/.7; ponderación|
|Hist boosting|LR log .01–.3; 80–300 iter.; 7–63 hojas; L2 log 1e-4–10; hoja mínima 10–60|
|MLP|32/64/100/(64,32); alpha log 1e-6–.1; LR log 1e-4–1e-2; 200/400 iter.; early stopping|
|GaussianNB|var_smoothing log 1e-12–1e-3|

Se ejecutan únicamente los espacios de las tres mejores familias de t+3.


In [ ]:
from src.datos import models, spaces
if HAS_DATA:
    tuned = execute("tuning")
    display(pd.DataFrame(tuned)[["horizon", "name", "cv_ap_mean", "cv_ap_std", "improvement", "improvement_exceeds_std", "params", "run_id"]])
    display(pd.read_csv(EVIDENCE / "tuning_trials.csv"))
    _ = execute("report")
    display(Image(filename=str(EVIDENCE / "tuning.png")))


La búsqueda de hiperparámetros y la selección usan la misma CV: puede haber optimismo de selección. El hold-out final permanece independiente y solo se abre después de congelar finalistas y umbrales.